In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd

import pickle
from pathlib import Path

In [ ]:
#persistent disk
data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-11-13_get_cohort_statistic"




In [ ]:
# This query represents dataset "viral disease" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8

def get_viral_condition_df():
    
    dataset_48844012_condition_sql = """
        SELECT
            c_occurrence.person_id,
            c_occurrence.condition_concept_id,
            c_standard_concept.concept_name as standard_concept_name,
            c_standard_concept.concept_code as standard_concept_code,
            c_standard_concept.vocabulary_id as standard_vocabulary,
            c_occurrence.condition_start_datetime,
            c_occurrence.condition_end_datetime,
            c_occurrence.condition_type_concept_id,
            c_type.concept_name as condition_type_concept_name,
            c_occurrence.stop_reason,
            c_occurrence.visit_occurrence_id,
            visit.concept_name as visit_occurrence_concept_name,
            c_occurrence.condition_source_value,
            c_occurrence.condition_source_concept_id,
            c_source_concept.concept_name as source_concept_name,
            c_source_concept.concept_code as source_concept_code,
            c_source_concept.vocabulary_id as source_vocabulary,
            c_occurrence.condition_status_source_value,
            c_occurrence.condition_status_concept_id,
            c_status.concept_name as condition_status_concept_name 
        FROM
            ( SELECT
                * 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
            WHERE
                (
                    condition_concept_id IN (SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (440029)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1)
                )  
                AND (
                    c_occurrence.PERSON_ID IN (SELECT
                        distinct person_id  
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                    WHERE
                        cb_search_person.person_id IN (SELECT
                            person_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                        WHERE
                            has_ehr_data = 1 ) 
                        AND cb_search_person.person_id IN (SELECT
                            person_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                        WHERE
                            has_whole_genome_variant = 1 
                        UNION
                        DISTINCT SELECT
                            person_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                        WHERE
                            has_lr_whole_genome_variant = 1 
                        UNION
                        DISTINCT SELECT
                            person_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                        WHERE
                            has_array_data = 1 ) 
                        AND cb_search_person.person_id IN (SELECT
                            criteria.person_id 
                        FROM
                            (SELECT
                                DISTINCT person_id, entry_date, concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                (concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (440029)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 )) criteria ) )
                    )
                ) c_occurrence 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                    ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                    ON c_occurrence.condition_type_concept_id = c_type.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                    ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                    ON v.visit_concept_id = visit.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                    ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                    ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

    df = pd.read_gbq(
            dataset_48844012_condition_sql,
            dialect="standard",
            use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
            progress_bar_type="tqdm_notebook")

    return df

In [ ]:
# This query represents dataset "kl;" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
    
def get_demographics_table():   
    dataset_08442758_person_sql = """
            SELECT
                person.person_id,
                person.gender_concept_id,
                p_gender_concept.concept_name as gender,
                person.birth_datetime as date_of_birth,
                person.race_concept_id,
                p_race_concept.concept_name as race,
                person.ethnicity_concept_id,
                p_ethnicity_concept.concept_name as ethnicity,
                person.sex_at_birth_concept_id,
                p_sex_at_birth_concept.concept_name as sex_at_birth,
                person.self_reported_category_concept_id,
                p_self_reported_category_concept.concept_name as self_reported_category 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
                    ON person.gender_concept_id = p_gender_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
                    ON person.race_concept_id = p_race_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
                    ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
                    ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
                    ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id"""

    dataset_08442758_person_df = pd.read_gbq(
            dataset_08442758_person_sql,
            dialect="standard",
            use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
            progress_bar_type="tqdm_notebook")
    
    return dataset_08442758_person_df

In [ ]:
 # This query represents dataset "kl;" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v8
    
def get_socioeconomic_table():

    dataset_91545369_zip_code_socioeconomic_sql = """
        SELECT
            observation.person_id,
            observation.observation_datetime,
            zip_code.zip3_as_string as zip_code,
            zip_code.fraction_assisted_income as assisted_income,
            zip_code.fraction_high_school_edu as high_school_education,
            zip_code.median_income,
            zip_code.fraction_no_health_ins as no_health_insurance,
            zip_code.fraction_poverty as poverty,
            zip_code.fraction_vacant_housing as vacant_housing,
            zip_code.deprivation_index,
            zip_code.acs as american_community_survey_year 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
        JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
                ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3 
                AND observation_source_concept_id = 1585250 
                AND observation.value_as_string NOT LIKE 'Res%'"""

    dataset_91545369_zip_code_socioeconomic_df = pd.read_gbq(
        dataset_91545369_zip_code_socioeconomic_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook")

    return dataset_91545369_zip_code_socioeconomic_df

In [ ]:
def create_df_pkl(df, directory):

    # 2) Choose a workspace folder for persistence
    out_file = Path(directory)

    # 3) Save the entire dict in one go
    with open(out_file, 'wb') as f:
        pickle.dump(df, f)

    print(f"Saved {len(df)} DataFrames to {out_file}")

In [ ]:
##function calls

cohort = get_viral_condition_df()
demo = get_demographics_table().drop_duplicates('person_id')
socio = get_socioeconomic_table().drop_duplicates('person_id')



create_df_pkl(cohort, (f"{data}/cohort_cond_df.pkl"))
create_df_pkl(demo, (f"{data}/cohort_demo_df.pkl"))
create_df_pkl(socio, (f"{data}/cohort_socio_df.pkl"))

In [ ]:
## Visualize 



